In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.api as sm

import sys
sys.path.append('..')  # adjust if your utils.py lives elsewhere
from utils import load_or_fetch  # or get_stock_data, whichever you used
from plot_setup import setup_chinese_font
setup_chinese_font()

# 1. Load 震元 data from cache
ZHENYUAN_CODE = 'sz.000705'  # <-- fill in your actual 浙江震元 baostock code
df = load_or_fetch(ZHENYUAN_CODE, '2024-01-01', '2024-12-31')

# 2. Compute volume ratio and next-day return
# Confirm the window matches Session 2. Handoff didn't pin it; 20 is the usual default.
VOL_WINDOW = 20

df['vol_ma'] = df['volume'].rolling(VOL_WINDOW).mean()
df['vol_ratio'] = df['volume'] / df['vol_ma']
df['ret'] = df['close'].pct_change()
df['next_day_ret'] = df['ret'].shift(-1)  # tomorrow's return on today's row

# 3. Build the x/y pair
mask = df['vol_ratio'].notna() & df['next_day_ret'].notna()
x_clean = df.loc[mask, 'vol_ratio']
y_clean = df.loc[mask, 'next_day_ret']

print(f"N = {len(x_clean)}")
print(f"Slope from np.polyfit (sanity check): {np.polyfit(x_clean, y_clean, 1)[0]:.6f}")

N = 222
Slope from np.polyfit (sanity check): 0.003860


In [5]:
X = sm.add_constant(x_clean)
results = sm.OLS(y_clean, X).fit()
print(results.summary())

                            OLS Regression Results                            
Dep. Variable:           next_day_ret   R-squared:                       0.015
Model:                            OLS   Adj. R-squared:                  0.011
Method:                 Least Squares   F-statistic:                     3.376
Date:                Tue, 21 Apr 2026   Prob (F-statistic):             0.0675
Time:                        15:09:58   Log-Likelihood:                 495.57
No. Observations:                 222   AIC:                            -987.1
Df Residuals:                     220   BIC:                            -980.3
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const         -0.0038      0.003     -1.361      0.1

In [6]:
from statsmodels.stats.diagnostic import het_breuschpagan

bp_stat, bp_pvalue, f_stat, f_pvalue = het_breuschpagan(results.resid, results.model.exog)

print(f"Breusch-Pagan LM statistic: {bp_stat:.4f}")
print(f"Breusch-Pagan p-value:     {bp_pvalue:.4f}")
print(f"F-statistic:               {f_stat:.4f}")
print(f"F p-value:                 {f_pvalue:.4f}")

Breusch-Pagan LM statistic: 5.9262
Breusch-Pagan p-value:     0.0149
F-statistic:               6.0338
F p-value:                 0.0148


In [7]:
results_robust = sm.OLS(y_clean, X).fit(cov_type='HC3')
print(results_robust.summary())

                            OLS Regression Results                            
Dep. Variable:           next_day_ret   R-squared:                       0.015
Model:                            OLS   Adj. R-squared:                  0.011
Method:                 Least Squares   F-statistic:                     1.593
Date:                Tue, 21 Apr 2026   Prob (F-statistic):              0.208
Time:                        15:15:35   Log-Likelihood:                 495.57
No. Observations:                 222   AIC:                            -987.1
Df Residuals:                     220   BIC:                            -980.3
Df Model:                           1                                         
Covariance Type:                  HC3                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const         -0.0038      0.003     -1.196      0.2

In [8]:
from statsmodels.stats.diagnostic import het_breuschpagan

ICBC_CODE = 'sh.601398'  # 工商银行; substitute your basket's code if different
df_icbc = load_or_fetch(ICBC_CODE, '2024-01-01', '2024-12-31')

df_icbc['vol_ma'] = df_icbc['volume'].rolling(VOL_WINDOW).mean()
df_icbc['vol_ratio'] = df_icbc['volume'] / df_icbc['vol_ma']
df_icbc['ret'] = df_icbc['close'].pct_change()
df_icbc['next_day_ret'] = df_icbc['ret'].shift(-1)

mask = df_icbc['vol_ratio'].notna() & df_icbc['next_day_ret'].notna()
x_icbc = df_icbc.loc[mask, 'vol_ratio']
y_icbc = df_icbc.loc[mask, 'next_day_ret']

X_icbc = sm.add_constant(x_icbc)

# Classical fit
results_icbc = sm.OLS(y_icbc, X_icbc).fit()
print("=" * 60)
print("工商银行 — Classical OLS")
print("=" * 60)
print(results_icbc.summary())

# Breusch-Pagan test
bp_stat, bp_pvalue, _, _ = het_breuschpagan(results_icbc.resid, results_icbc.model.exog)
print(f"\nBreusch-Pagan: LM = {bp_stat:.4f}, p = {bp_pvalue:.4f}")

# HC3 robust fit
results_icbc_robust = sm.OLS(y_icbc, X_icbc).fit(cov_type='HC3')
print("\n" + "=" * 60)
print("工商银行 — HC3 Robust")
print("=" * 60)
print(results_icbc_robust.summary())

工商银行 — Classical OLS
                            OLS Regression Results                            
Dep. Variable:           next_day_ret   R-squared:                       0.011
Model:                            OLS   Adj. R-squared:                  0.006
Method:                 Least Squares   F-statistic:                     2.419
Date:                Tue, 21 Apr 2026   Prob (F-statistic):              0.121
Time:                        15:36:03   Log-Likelihood:                 648.96
No. Observations:                 222   AIC:                            -1294.
Df Residuals:                     220   BIC:                            -1287.
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.0051      0.00